# Food delivery database: generating the data

This notebook fills an empty SQLite database (tables already created from the schema) with **fake, randomly generated data**. No data was downloaded from anywhere else.

Steps:
1. Upload the empty `food_delivery_db.db` file
2. Set up the lists of values the random data is picked from
3. Generate and insert Address, Customers, Restaurants and Menu_Items
4. Generate and insert Orders and Order_Items
5. Check the row counts and download the finished database

In [ ]:
# ---------- Step 1: imports and opening the database ----------
import sqlite3                      # to talk to the SQLite database
import random                       # to pick random values
import uuid                         # to make unique ids for the primary keys
from datetime import datetime, timedelta   # to make random dates
from google.colab import files      # to upload/download files in Colab

# Upload the empty database file from my computer
uploaded = files.upload()
db_name = list(uploaded.keys())[0]

conn = sqlite3.connect(db_name)

# SQLite does NOT check foreign keys unless this is switched on.
# Turning it on means any wrong link between tables gives an error instead of bad data.
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

## Step 2: Lists the random data comes from

In [ ]:
# ---------- Step 2: source lists ----------
# Names are picked from short lists. I mixed common UK names with names from

first_names = ["Oliver", "Amelia", "Jack", "Olivia", "Harry", "Isla", "George", "Mia", "Noah", "Emily",
               "Priya", "Rahul", "Aisha", "Daniel", "Sophie", "Ethan", "Zara", "Leo", "Grace", "Samuel"]
last_names = ["Smith", "Jones", "Taylor", "Brown", "Wilson", "Evans", "Patel", "Khan", "Clarke", "Walker",
              "Hughes", "Green", "Hall", "Wood", "Ali", "Singh", "Thomas", "Roberts", "Wright", "Turner"]

# Address data: everything is in the UK so the cities, streets and postcodes look realistic.
cities = ["London", "Manchester", "Birmingham", "Leeds", "Bristol", "Liverpool", "Glasgow", "Cardiff"]
streets = ["High Street", "Station Road", "Park Road", "Church Lane", "Victoria Road", "Mill Lane", "King Street"]
postcode_starts = ["E1", "N7", "SW9", "M14", "B15", "LS6", "BS8", "L17", "G12", "CF10"]

# NOMINAL data: categories with no order (cuisine type, payment method, category, dietary type)
cuisines = ["Italian", "Indian", "Chinese", "Mexican", "Japanese", "American", "Thai"]
rest_words = ["Golden", "Spice", "Little", "Royal", "Happy", "Urban", "Fresh", "Lucky"]
rest_types = ["Kitchen", "Grill", "House", "Bistro", "Takeaway", "Diner"]

# Each dish has (name, category, dietary type).
dishes = [("Margherita Pizza", "Main", "Vegetarian"), ("Chicken Curry", "Main", "Non-Vegetarian"),
          ("Veg Noodles", "Main", "Vegan"), ("Beef Burrito", "Main", "Non-Vegetarian"),
          ("Salmon Sushi", "Main", "Non-Vegetarian"), ("Cheeseburger", "Main", "Non-Vegetarian"),
          ("Pad Thai", "Main", "Non-Vegetarian"), ("Paneer Tikka", "Starter", "Vegetarian"),
          ("Spring Rolls", "Starter", "Vegan"), ("Garlic Bread", "Side", "Vegetarian"),
          ("Fries", "Side", "Vegan"), ("Rice", "Side", "Vegan"),
          ("Chocolate Brownie", "Dessert", "Vegetarian"), ("Ice Cream", "Dessert", "Vegetarian"),
          ("Cola", "Drink", "Vegan"), ("Mango Lassi", "Drink", "Vegetarian")]

# "Delivered" is repeated on purpose: random.choice picks each entry equally often,
# so repeating it makes most orders delivered (like a real service) and a few cancelled or in progress.
statuses = ["Delivered", "Delivered", "Delivered", "Delivered", "Cancelled", "Preparing", "Out for Delivery"]
payments = ["Credit Card", "Debit Card", "PayPal", "Apple Pay", "Cash"]   # only the method is stored, never card numbers (privacy)

def new_id():
    # UUIDs are used as primary keys because they are unique and cannot be guessed (unlike 1, 2, 3...)
    return str(uuid.uuid4())

def random_date(days_back):
    # a random moment between 1 and days_back days ago
    return datetime.now() - timedelta(days=random.randint(1, days_back))

## Step 3: Address, Customers, Restaurants and Menu_Items

In [ ]:
# ---------- Step 3: parent tables ----------
# These tables must be filled FIRST because other tables point to them with foreign keys.
num_customers = 500      # enough customers to give many different orders
num_restaurants = 60
num_orders = 1500        # the assignment needs at least 1000 rows in one table

addresses = []
customers = []
restaurants = []
menu_items = []
prices = {}              # menu item id -> price (the Menu_Items table has no price column, so it is kept here)

# --- Customers (each customer gets their own address) ---
for i in range(num_customers):
    addr_id = new_id()
    addresses.append((addr_id, "United Kingdom", random.choice(cities),
                      str(random.randint(1, 200)) + " " + random.choice(streets),
                      random.choice(postcode_starts) + " " + str(random.randint(1, 9)) + "AB"))
    fn = random.choice(first_names)
    ln = random.choice(last_names)
    # The email has a number (i) at the end so no two emails are the same (the email column is UNIQUE).
    # example.com is a domain made for examples, so these can never be a real person's email.
    email = fn.lower() + "." + ln.lower() + str(i) + "@example.com"
    mobile = "07" + str(random.randint(100000000, 999999999))
    reg = random_date(1000).strftime("%Y-%m-%d")      # registered within about the last 3 years
    customers.append((new_id(), fn, ln, email, mobile, reg, addr_id))

# --- Restaurants and their menus ---
for i in range(num_restaurants):
    addr_id = new_id()
    addresses.append((addr_id, "United Kingdom", random.choice(cities),
                      str(random.randint(1, 200)) + " " + random.choice(streets),
                      random.choice(postcode_starts) + " " + str(random.randint(1, 9)) + "CD"))
    rest_id = new_id()
    # The number at the end keeps every restaurant name different
    name = random.choice(rest_words) + " " + random.choice(rest_types) + " " + str(i + 1)
    # ORDINAL data: rating from 3 to 5 (higher is better, but the gaps are not exactly equal)
    # INTERVAL data: opening year (differences make sense, but year 0 is not a real "start")
    restaurants.append((rest_id, name, random.choice(cuisines), round(random.uniform(3, 5), 1),
                        addr_id, str(random.randint(1995, 2024))))

    # random.sample picks 8 to 12 DIFFERENT dishes, so a menu has no repeated dish
    for d in random.sample(dishes, random.randint(8, 12)):
        item_id = new_id()
        # availability is 1 (available) for all items; preparation time is in minutes
        menu_items.append((item_id, rest_id, d[0], d[1], d[2], 1,
                           str(random.choice([10, 15, 20, 30])) + " minutes"))
        # RATIO data: a price has a real zero, so "twice as expensive" makes sense
        prices[item_id] = round(random.uniform(2, 15), 2)

# Insert in this order: Address first, because Customers and Restaurants refer to it.
cur.executemany("INSERT INTO Address VALUES (?,?,?,?,?)", addresses)
cur.executemany("INSERT INTO Customers VALUES (?,?,?,?,?,?,?)", customers)
cur.executemany("INSERT INTO Restaurants VALUES (?,?,?,?,?,?)", restaurants)
cur.executemany("INSERT INTO Menu_Items VALUES (?,?,?,?,?,?,?)", menu_items)
conn.commit()

## Step 4: Orders and Order_Items

In [ ]:
# ---------- Step 4: child tables ----------
# Orders and Order_Items depend on everything above, so they are filled LAST.
orders = []
order_items = []

for i in range(num_orders):
    order_id = new_id()
    customer = random.choice(customers)
    restaurant = random.choice(restaurants)

    # Only use items from THIS restaurant's menu, so an order never contains food the restaurant does not sell
    items = [m for m in menu_items if m[1] == restaurant[0]]
    chosen = random.sample(items, random.randint(1, 4))      # 1 to 4 different items per order

    # Order_Items has a compound primary key (order_id, item_id), so each item can only appear once per order.
    # random.sample already gives different items, and the quantity says how many of that item.
    total = 0
    for item in chosen:
        qty = random.randint(1, 3)
        price = prices[item[0]]
        total += qty * price
        order_items.append((order_id, item[0], qty, price))   # price is saved now, so old orders keep their price

    status = random.choice(statuses)

    # INTERVAL data: temperature in degrees Celsius (0 degrees is not "no heat").
    # Only delivered orders get a value; the others stay empty (NULL) because there is nothing to measure yet.
    temp = round(random.uniform(50, 65), 1) if status == "Delivered" else None

    # The order must happen AFTER the customer registered, so the date is picked
    # between the registration date and now.
    reg_date = datetime.strptime(customer[5], "%Y-%m-%d")
    seconds_since_reg = int((datetime.now() - reg_date).total_seconds())
    order_date = reg_date + timedelta(seconds=random.randint(0, seconds_since_reg))

    # total_amount is the sum of its order items, so the Orders table always agrees with Order_Items.
    # RATIO data: total_amount
    orders.append((order_id, customer[0], restaurant[0], order_date.strftime("%Y-%m-%d %H:%M:%S"),
                   status, random.choice(payments), temp, round(total, 2)))

# Orders must go in before Order_Items, because Order_Items points to Orders.
cur.executemany("INSERT INTO Orders VALUES (?,?,?,?,?,?,?,?)", orders)
cur.executemany("INSERT INTO Order_Items VALUES (?,?,?,?)", order_items)
conn.commit()

## Step 5: Check the result and download

In [ ]:
# ---------- Step 5: check the row counts ----------
total_rows = 0
for table in ["Address", "Customers", "Restaurants", "Menu_Items", "Orders", "Order_Items"]:
    count = cur.execute("SELECT COUNT(*) FROM " + table).fetchone()[0]
    total_rows += count
    print(table, count)
print("Total rows:", total_rows)

# Returns an empty list if every foreign key points to a real row
print("Foreign key problems:", cur.execute("PRAGMA foreign_key_check").fetchall())

In [ ]:
# Close the connection and download the finished database
conn.close()
files.download(db_name)